# 01 · Published checkpoints on dev, test and the no-speech clips

Runs on Kaggle with a GPU. **Settings → Accelerator: GPU, Internet: On.** No manual uploads: the
notebook clones the public repository, which contains the dataset (`data/`).

Every system in `configs/baselines.yaml` with `trained_by_us: false` and `status: planned` is run by
`scripts/infer_pretrained.py`, one process per system, in a virtual environment with the pinned
packages of `requirements/baselines.txt`. Checkpoints are pinned to a commit SHA (written to
`results/predictions/<system>.meta.json`). Scoring is not done here: `scripts/score.py` scores all
systems together, with `requirements/data.txt`.

`TRIAL = True` runs 8 clips per split into `/kaggle/working/trial` (a short test of every system).

Output: `/kaggle/working/predictions_baselines.zip` with `results/predictions/`. Unzip it at the
repository root.

In [ ]:
REPO_URL = "https://github.com/eneserman/NLP.git"  # the public repository
BRANCH = "main"
TRIAL = False
BATCH_SIZE = 16

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded")
except Exception:
    print("No HF_TOKEN secret; using anonymous access")

In [ ]:
# The repository and a virtual environment with only our pinned packages live in /tmp, so that the
# preinstalled packages of the Kaggle image cannot conflict and /kaggle/working holds only the results.
import os
os.environ.pop("PYTHONPATH", None)          # nothing from the Kaggle image on the path of our environment
%cd /tmp
!rm -rf repo venv && git clone -q -b $BRANCH $REPO_URL repo
%cd /tmp/repo
!git log -1 --format='commit %H'
!pip install -q virtualenv==21.15.9 && python -m virtualenv -q /tmp/venv
PY = "/tmp/venv/bin/python"
!{PY} -m pip install -q -r requirements/baselines.txt
!{PY} -m pip freeze > /kaggle/working/pip_freeze.txt && cat /kaggle/working/pip_freeze.txt
!{PY} -c "import sys; print('sys.path:', sys.path)"
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "no GPU"

Tests of the inference code, with the real torch and transformers of the environment.

In [ ]:
!{PY} -m pytest -q tests/test_inference.py

In [ ]:
!{PY} scripts/unpack_audio.py

In [ ]:
import subprocess, yaml
listed = yaml.safe_load(open("configs/baselines.yaml", encoding="utf-8"))["baselines"]
systems = [b["system"] for b in listed if not b.get("trained_by_us") and b.get("status") == "planned"]
print(systems)
extra = ["--limit", "8", "--out-dir", "/kaggle/working/trial"] if TRIAL else []
failed = []
for s in systems:
    print("=" * 30, s, flush=True)
    r = subprocess.run([PY, "scripts/infer_pretrained.py", "--system", s, "--batch-size", str(BATCH_SIZE), *extra])
    if r.returncode != 0:
        failed.append(s)
print("failed:", failed)

In [ ]:
if TRIAL:
    !for f in /kaggle/working/trial/*.test.jsonl; do echo "== $f"; head -n 3 "$f"; done
    !cat /kaggle/working/trial/*.meta.json
else:
    !zip -q -r /kaggle/working/predictions_baselines.zip results/predictions && ls -lh /kaggle/working